In [ ]:
'''
notebook for filling the missing values in a variable using GLODAPv2_2016b_MappedClimatologies
'''


In [ ]:
import warnings
warnings.filterwarnings('ignore') # don't output warnings
import xarray as xr
import numpy as np
import pandas as pd 
from modules.module_data_postprocessing import nearest_valid_point
from tqdm import tqdm

========================================================================================================================================================

User configs:

In [ ]:
dir_data = '/space/hall7/sitestore/eccc/crd/cccma/users/rpg002/data'
dir_glodap_clim = '/space/hall7/sitestore/eccc/crd/cccma/users/rpg002/data/GLODAP/GLODAPv2_2016b_MappedClimatologies'

var_list = ['silicate', 'NO3', 'PO4']



========================================================================================================================================================

In [ ]:
def extract_model_grid_within_distance(ref, data,mask = None, lev_bins = None, tol=2,thresh=160,badval=-999999 ):
    
    model_lat = data['lat']
    model_lon = data['lon']

    if len(model_lat.shape) == 1:
        Y, X = np.meshgrid(
                model_lat['lat'].values,
                model_lon['lon'].values,
                indexing='ij')
    if mask is not None:
        assert Y.shape == mask.shape
    else:
        mask =1 
    
    model_lat_ind, model_lon_ind = nearest_valid_point(ref['lat'], ref['lon'] ,Y, X, mask = mask,tol=tol,thresh=thresh,badval=badval )
    ref = ref[model_lat_ind > badval]

    model_lat_ind = model_lat_ind[model_lat_ind>badval]
    model_lon_ind = model_lon_ind[model_lon_ind>badval]
    ref['model_lat'] = Y[model_lat_ind, model_lon_ind]
    ref['model_lon'] = X[model_lat_ind, model_lon_ind]


    return ref



In [ ]:

dict_filled = {}
for var in tqdm(var_list):
    if var.lower() not in dict_filled.keys():
        print(var)
        df = pd.read_csv(f'{dir_data}/{var.lower()}/observation/GLODAP/GLODAP_v2_2023_{var.lower()}_1984-2021.csv')
        df.loc[df['lon'] > 180, 'lon'] = df.loc[df['lon'] > 180, 'lon'] - 360
        clim = xr.open_dataset(f'{dir_glodap_clim}/GLODAPv2.2016b.{var}.nc')
        clim = clim.rename({'depth_surface':'depth'}).assign_coords(depth = clim['Depth'].values)[var]
        clim['lon'] = np.mod(clim['lon'],360)
        clim = clim.sortby('lon')
        clim['lon'] = ((clim['lon'] + 180) % 360) - 180
        clim = clim.sortby('lon')
        mask = clim[0].where(np.isnan(clim[0]),1).fillna(0)
        
        df_ = extract_model_grid_within_distance(df, clim, mask = mask)
        df_ = df_[np.isnan(df_[var.lower()])]
        df_fill = np.array([clim.sel(lat = df_['model_lat'].values[i], lon = df_['model_lon'].values[i], depth =  df_['lev'].values[i], method = 'nearest') 
                        for i in range(len(df_))])
        df_[var.lower()] = df_fill

        df_merged = df.merge(
                df_[[i for i in df_.columns if i not in ['model_lat', 'model_lon']]],  # only the columns you want to use for filling
                on=[i for i in df_.columns if i not in ['model_lat', 'model_lon', var.lower()]],
                how='left',
                suffixes=('', '_fill'))

        df_merged[var.lower()] = df_merged[var.lower()].fillna(df_merged[f'{var.lower()}_fill'])
        df_filled = df_merged.drop(columns=f'{var.lower()}_fill')

        dict_filled[var.lower()] = df_filled



In [ ]:
for var in dict_filled.keys():
    dict_filled[var].to_csv( f'{dir_data}/{var}/observation/GLODAP/GLODAP_v2_2023_{var}_1984-2021.csv')
        